# U1-01 · Python, NumPy, matemáticas y estadística con los pingüinos

*Especialista en Inteligencia Artificial (IFCD107) · Unidad 1*

Ejecuta cada celda con **Mayús + Intro**. Todo funciona en el navegador; la primera vez tarda unos segundos en cargar las bibliotecas.

> Datos: palmerpenguins, de Horst, Hill y Gorman (2020), CC0. Gorman, K. B., Williams, T. D. y Fraser, W. R. (2014). *PLoS ONE*, 9(3), e90081. https://doi.org/10.1371/journal.pone.0090081

## 1.3 · Python desde cero

`b1_basico.py`

In [ ]:
masa = 3750                    # int: número entero (gramos)
pico = 39.1                    # float: número decimal (mm)
especie = "Adelie"             # str: texto
macho = True                   # bool: verdadero o falso
print(type(masa).__name__, type(pico).__name__,
      type(especie).__name__, type(macho).__name__)
print(f"{especie}: {masa / 1000:.2f} kg, pico de {pico} mm")
print(masa // 1000, masa % 1000, 2 ** 10, 7 / 2)  # //: entera
print("Ade" in especie, especie.upper(), especie[:3], len(especie))
# condiciones
if masa > 4500:
    tamaño = "grande"
elif masa > 3500:
    tamaño = "mediano"
else:
    tamaño = "pequeño"
print("tamaño:", tamaño)
# bucles
for año in range(2007, 2010):
    print(año, end=" ")
print()
n = 0
while 2 ** n < masa:           # cuántas veces hay que doblar 1 g
    n += 1
print("2 elevado a", n, "supera", masa)

`b2_estructuras.py`

In [ ]:
import csv

with open("penguins.csv") as f:
    filas = list(csv.DictReader(f))           # lista de diccionarios
print(len(filas), "filas; primera:", filas[0]["species"],
      filas[0]["island"])
islas = {f["island"] for f in filas}          # conjunto: sin repetir
print("islas:", sorted(islas))
cuenta = {}                             # especie -> número
for f in filas:
    cuenta[f["species"]] = cuenta.get(f["species"], 0) + 1
print("pingüinos por especie:", cuenta)
masas = [int(f["body_mass_g"]) for f in filas
         if f["body_mass_g"] != "NA"]          # lista por comprensión
print(f"masa: mínima {min(masas)}, máxima {max(masas)}, "
      f"media {sum(masas) / len(masas):.1f} g")
medida = (39.1, 18.7, 181)                    # tupla: no se modifica
largo, alto, aleta = medida                   # desempaquetar
print("pico", largo, "x", alto, "| aleta", aleta)
por_isla = {}
for f in filas:
    por_isla.setdefault(f["island"], set()).add(f["species"])
for isla, especies in sorted(por_isla.items()):
    print(f"  {isla}: {', '.join(sorted(especies))}")

`b3_funciones.py`

In [ ]:
import csv

def leer(ruta, columna):
    """Devuelve los valores numéricos de una columna, sin los NA."""
    with open(ruta) as f:
        return [float(fila[columna]) for fila in csv.DictReader(f)
                if fila[columna] != "NA"]

def media(valores):
    if not valores:
        raise ValueError("no hay valores para promediar")
    return sum(valores) / len(valores)

def desviacion(valores, muestral=True):    # argumento por omisión
    m = media(valores)
    n = len(valores) - (1 if muestral else 0)
    return (sum((v - m) ** 2 for v in valores) / n) ** 0.5

aleta = leer("penguins.csv", "flipper_length_mm")
print(f"aleta: media {media(aleta):.2f} mm, desviación "
      f"{desviacion(aleta):.2f} (poblacional "
      f"{desviacion(aleta, muestral=False):.2f})")
try:
    media([])
except ValueError as e:
    print("error controlado:", e)
try:
    leer("penguins.csv", "peso")
except KeyError as e:
    print("columna inexistente:", e)
assert abs(media([1, 2, 3]) - 2) < 1e-12   # una prueba mínima

`b4_clases.py`

In [ ]:
from dataclasses import dataclass

@dataclass
class Pinguino:
    especie: str
    pico_mm: float
    aleta_mm: float
    masa_g: int

    def masa_kg(self):
        return self.masa_g / 1000

    def es_grande(self, umbral=4500):
        return self.masa_g > umbral

class Colonia:
    def __init__(self, isla):
        self.isla, self.miembros = isla, []
    def añadir(self, p):
        self.miembros.append(p)
    def __len__(self):
        return len(self.miembros)
    def media_masa(self):
        return sum(p.masa_g for p in self) / len(self)
    def __iter__(self):
        return iter(self.miembros)

c = Colonia("Biscoe")
c.añadir(Pinguino("Gentoo", 46.1, 211, 4500))
c.añadir(Pinguino("Adelie", 37.8, 174, 3400))
c.añadir(Pinguino("Gentoo", 50.0, 230, 5700))
print(c.miembros[0])
print(len(c), "pingüinos; media", round(c.media_masa()), "g;",
      "grandes:", [p.especie for p in c if p.es_grande()])

`b5_ficheros.py`

In [ ]:
import csv, json
from pathlib import Path

carpeta = Path("resultados")
carpeta.mkdir(exist_ok=True)               # crea la carpeta si falta
resumen = {}
with open("penguins.csv") as f:
    for fila in csv.DictReader(f):
        e = fila["species"]
        if fila["body_mass_g"] == "NA":
            continue
        r = resumen.setdefault(e, {"n": 0, "masa_total": 0})
        r["n"] += 1
        r["masa_total"] += int(fila["body_mass_g"])
for e, r in resumen.items():
    r["masa_media"] = round(r.pop("masa_total") / r["n"], 1)
ruta = carpeta / "resumen.json"
ruta.write_text(json.dumps(resumen, indent=2, ensure_ascii=False))
print(ruta, ruta.stat().st_size, "bytes")
leido = json.loads(ruta.read_text())       # se vuelve a leer igual
print(leido == resumen, leido["Gentoo"])
print([p.name for p in carpeta.iterdir()])

## 1.4 · Matemáticas para la IA con Python

`c0_numpy.py`

In [ ]:
import time
import numpy as np

a = np.array([39.1, 39.5, 40.3, 36.7])       # de una lista
print(a.dtype, a.shape, a.mean().round(3), a[1:3], a[a > 39])
M = np.arange(12).reshape(3, 4)              # matriz 3 x 4
print(M, M.shape, M.sum(axis=0), M.T.shape, sep="\n")
print(M * 10 + np.array([1, 2, 3, 4]))       # difusión por filas
rng = np.random.default_rng(0)
x = rng.normal(size=1_000_000)
t = time.perf_counter()
s = 0.0
for v in x:                                  # bucle de Python
    s += v * v
t1 = time.perf_counter() - t
t = time.perf_counter()
s2 = (x * x).sum()                           # vectorizado
t2 = time.perf_counter() - t
print(f"bucle {t1:.3f} s, NumPy {t2:.4f} s, "
      f"{t1 / t2:.0f} veces más rápido; iguales: {np.isclose(s, s2)}")

`c1_vectores.py`

In [ ]:
import numpy as np
import pandas as pd

p = pd.read_csv("penguins.csv").dropna()
cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm"]
a = p[cols].iloc[0].to_numpy()             # un Adelie
g = p[cols].iloc[200].to_numpy()           # otro pingüino
print(p.species.iloc[0], a, "|", p.species.iloc[200], g)
print("suma:", a + g, "| escalar x 2:", 2 * a)
print("producto escalar:", a @ g, "=", (a * g).sum())
print(f"norma (longitud): {np.linalg.norm(a):.2f}")
print(f"distancia euclídea: {np.linalg.norm(a - g):.2f}")
cos = a @ g / (np.linalg.norm(a) * np.linalg.norm(g))
print(f"coseno del ángulo: {cos:.4f}")
# con las variables estandarizadas la distancia es más justa
Z = (p[cols] - p[cols].mean()) / p[cols].std()
za, zg = Z.iloc[0].to_numpy(), Z.iloc[200].to_numpy()
print("estandarizados:", za.round(2), zg.round(2))
print(f"distancia estandarizada: {np.linalg.norm(za - zg):.2f}")

`c2_matrices.py`

In [ ]:
import numpy as np
import pandas as pd

np.set_printoptions(suppress=True)
p = pd.read_csv("penguins.csv").dropna()
y = p.body_mass_g.to_numpy(float)           # masa (g)
X = np.column_stack([np.ones(len(p)),       # columna de unos
                     p.flipper_length_mm])  # aleta (mm)
print("X:", X.shape, "| y:", y.shape)
A = X.T @ X                                 # matriz 2 x 2
b = X.T @ y
print("X'X =\n", A.round(0))
beta = np.linalg.solve(A, b)                # ecuaciones normales
print("b0, b1 =", beta.round(3))
beta2, *_ = np.linalg.lstsq(X, y, rcond=None)
print("con lstsq:", beta2.round(3))
print("determinante de X'X:", f"{np.linalg.det(A):.3e}")
pred = X @ beta
print(f"error cuadrático medio: {np.mean((y - pred) ** 2):.0f} g²")
print("previsión para una aleta de 200 mm:",
      round(beta @ [1, 200]), "g")

`c3_derivadas.py`

In [ ]:
import numpy as np

def f(x):
    return x ** 3 - 2 * x                    # una función cualquiera
def df(x):
    return 3 * x ** 2 - 2                    # su derivada exacta

for h in [1e-1, 1e-3, 1e-5, 1e-9, 1e-13]:
    aprox = (f(1 + h) - f(1 - h)) / (2 * h)   # diferencia centrada
    print(f"h={h:.0e}: derivada en 1 = {aprox:.10f}"
          f"  error {abs(aprox - df(1)):.1e}")

def J(w):                                   # función de dos variables
    return (w[0] - 3) ** 2 + 10 * (w[1] + 1) ** 2
def gradiente(F, w, h=1e-6):
    g = np.zeros_like(w)
    for i in range(len(w)):
        e = np.zeros_like(w); e[i] = h
        g[i] = (F(w + e) - F(w - e)) / (2 * h)
    return g
w = np.array([0.0, 0.0])
print("gradiente en (0, 0):", gradiente(J, w).round(4),
      "| exacto: [-6, 20]")

`c4_descenso.py`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

p = pd.read_csv("penguins.csv").dropna()
x = ((p.flipper_length_mm - p.flipper_length_mm.mean())
     / p.flipper_length_mm.std()).to_numpy()   # aleta estandarizada
y = p.body_mass_g.to_numpy() / 1000             # masa en kg
fig, eje = plt.subplots(figsize=(4.4, 2.0))
for tasa, estilo in [(0.01, "-"), (0.1, "--"), (0.9, ":"),
                     (1.05, "-.")]:
    b0 = b1 = 0.0
    perdida = []
    for paso in range(60):
        e = b0 + b1 * x - y                     # errores
        perdida.append(np.mean(e ** 2))
        b0 -= tasa * 2 * e.mean()               # derivadas de la
        b1 -= tasa * 2 * (e * x).mean()         # pérdida media
    print(f"tasa {tasa}: b0={b0:.3f}, b1={b1:.3f},"
          f" pérdida final {perdida[-1]:.4f}")
    eje.plot(perdida, estilo, label=f"tasa {tasa}")
eje.set_yscale("log"); eje.set_ylim(0.1, 30)
eje.set_xlabel("paso"); eje.set_ylabel("error cuadrático medio")
eje.legend(fontsize=6); plt.show()
b1, b0 = np.polyfit(x, y, 1)
print(f"solución exacta: b0={b0:.3f}, b1={b1:.3f}")

`c10_logistica.py`

In [ ]:
import numpy as np
import pandas as pd

p = pd.read_csv("penguins.csv").dropna()
a = p[p.species != "Chinstrap"]              # Adelia frente a papúa
x = ((a.bill_depth_mm - a.bill_depth_mm.mean())
     / a.bill_depth_mm.std()).to_numpy()
y = (a.species == "Gentoo").to_numpy(float)  # 1 = papúa
sig = lambda z: 1 / (1 + np.exp(-z))
w = b = 0.0
for paso in range(501):
    q = sig(w * x + b)                       # probabilidad de papúa
    perdida = -np.mean(y * np.log(q) + (1 - y) * np.log(1 - q))
    w -= 0.5 * np.mean((q - y) * x)          # gradiente de la
    b -= 0.5 * np.mean(q - y)                # entropía cruzada
    if paso in (0, 10, 100, 500):
        print(f"paso {paso:3d}: pérdida {perdida:.4f}, "
              f"acierto {np.mean((q > 0.5) == y):.3f}")
print(f"w = {w:.3f}, b = {b:.3f}")
corte = a.bill_depth_mm.mean() - b / w * a.bill_depth_mm.std()
print(f"frontera: alto del pico = {corte:.2f} mm")

`c5_azar.py`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)              # semilla: reproducible
dado = rng.integers(1, 7, size=100_000)
for n in [10, 100, 10_000, 100_000]:    # ley de grandes números
    print(f"{n:>7} tiradas: media {dado[:n].mean():.4f}"
          f"  (teórica 3.5)")
# teorema central del límite: medias de 30 tiradas
medias = rng.integers(1, 7, size=(20_000, 30)).mean(axis=1)
print(f"medias de 30 tiradas: media {medias.mean():.3f},"
      f" desviación {medias.std():.4f}"
      f" (teórica {np.sqrt(35 / 12 / 30):.4f})")
fig, (a, b) = plt.subplots(1, 2, figsize=(4.4, 1.7),
                           layout="constrained")
a.hist(dado, bins=np.arange(0.5, 7.5), rwidth=0.8)
a.set_title("Una tirada")
b.hist(medias, bins=np.arange(2.2, 4.8, 1 / 30) + 1 / 60)
b.set_title("Media de 30 tiradas")
plt.show()

`c6_bayes.py`

In [ ]:
import pandas as pd

p = pd.read_csv("penguins.csv").dropna(subset=["flipper_length_mm"])
larga = p.flipper_length_mm > 210              # el «indicio»
gentoo = p.species == "Gentoo"
P_G = gentoo.mean()                            # a priori
P_L_dado_G = larga[gentoo].mean()              # verosimilitud
P_L = larga.mean()                             # evidencia
print(f"P(Gentoo) = {P_G:.3f}")
print(f"P(aleta > 210 | Gentoo) = {P_L_dado_G:.3f}")
print(f"P(aleta > 210) = {P_L:.3f}")
print(f"Bayes: P(Gentoo | aleta > 210) = "
      f"{P_L_dado_G * P_G / P_L:.3f}")
print(f"contando directamente:          "
      f"{gentoo[larga].mean():.3f}")
print(f"P(aleta > 210 | no Gentoo) = {larga[~gentoo].mean():.3f}")

`c11_entropia.py`

In [ ]:
import numpy as np
import pandas as pd

def entropia(etiquetas):
    p = etiquetas.value_counts(normalize=True).to_numpy()
    return -(p * np.log2(p)).sum()           # bits

p = pd.read_csv("penguins.csv").dropna(subset=["flipper_length_mm"])
H = entropia(p.species)
print(f"entropía de la especie: {H:.3f} bits"
      f" (máximo {np.log2(3):.3f})")
for nombre, corte in [("aleta > 206", p.flipper_length_mm > 206),
                      ("pico > 44", p.bill_length_mm > 44),
                      ("isla Biscoe", p.island == "Biscoe")]:
    resto = sum(len(g) / len(p) * entropia(g.species)
                for _, g in p.groupby(corte))
    print(f"{nombre:<12} entropía después {resto:.3f},"
          f" ganancia {H - resto:.3f} bits")

`c7_normal.py`

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

p = pd.read_csv("penguins.csv").dropna()
m = p[(p.species == "Gentoo") & (p.sex == "male")].body_mass_g
mu, s = m.mean(), m.std()
print(f"Gentoo machos: n={len(m)}, media {mu:.0f} g, desv. {s:.0f} g")
normal = stats.norm(mu, s)
for a, b in [(mu - s, mu + s), (mu - 2 * s, mu + 2 * s)]:
    real = ((m >= a) & (m <= b)).mean()
    teo = normal.cdf(b) - normal.cdf(a)
    print(f"entre {a:.0f} y {b:.0f} g: normal {teo:.3f},"
          f" observado {real:.3f}")
print(f"P(masa > 6000 g) según la normal: {normal.sf(6000):.4f};"
      f" observados {(m > 6000).sum()} de {len(m)}")
w = stats.shapiro(m)
print(f"prueba de Shapiro-Wilk: p = {w.pvalue:.3f}")

`c8_autovalores.py`

In [ ]:
import numpy as np
import pandas as pd

p = pd.read_csv("penguins.csv").dropna()
cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm",
        "body_mass_g"]
Z = ((p[cols] - p[cols].mean()) / p[cols].std()).to_numpy()
C = Z.T @ Z / (len(Z) - 1)                   # matriz de correlaciones
val, vec = np.linalg.eigh(C)                 # autovalores y vectores
orden = val.argsort()[::-1]
val, vec = val[orden], vec[:, orden]
print("autovalores:", val.round(3), "| suman", val.sum().round(3))
print("varianza explicada:", (val / val.sum()).round(3))
print("primer autovector:", vec[:, 0].round(3))
v = vec[:, 0]
print("C v = λ v:", np.allclose(C @ v, val[0] * v))
proy = Z @ vec[:, :2]                        # dos nuevas variables
print("medias de la 1.ª componente por especie:",
      pd.Series(proy[:, 0]).groupby(p.species.to_numpy()).mean()
      .round(2).to_dict())

In [ ]:
del C   # libera el nombre C, que usan las fórmulas de statsmodels

## 1.5 · Estadística con Python

`d1_descriptiva.py`

In [ ]:
import pandas as pd

p = pd.read_csv("penguins.csv")
print("faltan:", p.isna().sum()[p.isna().sum() > 0].to_dict())
m = p.body_mass_g
print(f"masa: media {m.mean():.1f}, mediana {m.median():.0f},"
      f" moda {m.mode()[0]:.0f}")
print(f"desviación {m.std():.1f}, rango intercuartílico "
      f"{m.quantile(.75) - m.quantile(.25):.0f},"
      f" asimetría {m.skew():.3f}")
tabla = p.groupby("species").body_mass_g.agg(
    ["count", "mean", "std", "min", "median", "max"]).round(0)
print(tabla.to_string())
cv = p.groupby("species").body_mass_g.std() / \
    p.groupby("species").body_mass_g.mean()
print("coeficiente de variación:", cv.round(3).to_dict())

`d2_simpson.py`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

p = pd.read_csv("penguins.csv").dropna(
    subset=["bill_length_mm", "bill_depth_mm"])
r = p.bill_length_mm.corr(p.bill_depth_mm)
print(f"todos juntos: correlación largo-alto del pico {r:.3f}")
for e, g in p.groupby("species"):
    print(f"  {e:<10} {g.bill_length_mm.corr(g.bill_depth_mm):.3f}"
          f"  (n={len(g)})")
fig, eje = plt.subplots(figsize=(4.4, 2.6))
for e, g in p.groupby("species"):
    eje.scatter(g.bill_length_mm, g.bill_depth_mm, s=5, label=e)
eje.set_xlabel("largo del pico (mm)")
eje.set_ylabel("alto del pico (mm)"); eje.legend()
plt.show()

`d3_contraste.py`

In [ ]:
import pandas as pd
from scipy import stats

p = pd.read_csv("penguins.csv").dropna()
a = p[p.species == "Adelie"]
mach = a[a.sex == "male"].body_mass_g
hemb = a[a.sex == "female"].body_mass_g
t = stats.ttest_ind(mach, hemb, equal_var=False)   # t de Welch
print(f"Adelie: machos {mach.mean():.0f} g, hembras "
      f"{hemb.mean():.0f} g, diferencia "
      f"{mach.mean() - hemb.mean():.0f} g")
print(f"t = {t.statistic:.2f}, p = {t.pvalue:.2e}")
# ¿depende la especie de la isla? tabla de contingencia
tabla = pd.crosstab(p.island, p.species)
chi = stats.chi2_contingency(tabla)
print(tabla.to_string())
print(f"chi cuadrado = {chi.statistic:.1f}, gl = {chi.dof},"
      f" p = {chi.pvalue:.1e}")
# mismo contraste en una isla con dos especies y años distintos
d = p[p.island == "Dream"]
r = stats.chi2_contingency(pd.crosstab(d.year, d.species))
print(f"Dream, especie según el año: p = {r.pvalue:.3f}")

`d5_ols.py`

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

p = pd.read_csv("penguins.csv").dropna()
m = smf.ols("body_mass_g ~ flipper_length_mm + C(species) + C(sex)",
            data=p).fit()
tabla = pd.DataFrame({"coef": m.params, "err. est.": m.bse,
                      "p": m.pvalues}).round(3)
tabla.index = ["constante", "Chinstrap", "Gentoo", "macho", "aleta"]
print(tabla.to_string())
print(f"R² = {m.rsquared:.3f}, error típico residual "
      f"{m.mse_resid ** 0.5:.0f} g, n = {int(m.nobs)}")
ic = m.conf_int().loc["flipper_length_mm"].round(1).tolist()
print("IC 95 % del efecto de 1 mm de aleta:", ic, "g")

`d4_intervalo.py`

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

p = pd.read_csv("penguins.csv").dropna()
c = p[p.species == "Chinstrap"].bill_length_mm.to_numpy()
n, m, s = len(c), c.mean(), c.std(ddof=1)
t = stats.t.ppf(0.975, n - 1)
print(f"Chinstrap: n={n}, media del pico {m:.2f} mm")
print(f"IC 95 % con la t de Student: {m - t * s / np.sqrt(n):.2f}"
      f" a {m + t * s / np.sqrt(n):.2f}")
rng = np.random.default_rng(0)
boot = [rng.choice(c, n).mean() for _ in range(10_000)]
print(f"IC 95 % bootstrap:           "
      f"{np.percentile(boot, 2.5):.2f} a "
      f"{np.percentile(boot, 97.5):.2f}")
# ¿cuántos de 1000 intervalos de muestras de 20 contienen la media?
poblacion = p.bill_length_mm.to_numpy(); mu = poblacion.mean()
dentro = 0
for _ in range(1000):
    x = rng.choice(poblacion, 20, replace=False)
    h = stats.t.ppf(0.975, 19) * x.std(ddof=1) / np.sqrt(20)
    dentro += x.mean() - h <= mu <= x.mean() + h
print(f"intervalos que contienen la media: {dentro} de 1000")

`d6_permutacion.py`

In [ ]:
import numpy as np
import pandas as pd

p = pd.read_csv("penguins.csv").dropna()
c = p[p.species == "Chinstrap"]
y07 = c[c.year == 2007].flipper_length_mm.to_numpy()
y09 = c[c.year == 2009].flipper_length_mm.to_numpy()
obs = y09.mean() - y07.mean()
print(f"aleta media de los barbijos: 2007 {y07.mean():.2f} mm,"
      f" 2009 {y09.mean():.2f} mm; diferencia {obs:.2f}")
todos = np.concatenate([y07, y09])
rng = np.random.default_rng(0)
difs = []
for _ in range(10_000):                      # si el año no importara,
    rng.shuffle(todos)                       # sus etiquetas serían
    difs.append(todos[len(y07):].mean()      # intercambiables
                - todos[:len(y07)].mean())
pval = np.mean(np.abs(difs) >= abs(obs))
print(f"diferencias tan grandes al azar: {pval:.4f} de las veces")

**Prueba.** En `c4_descenso.py`, cambia la tasa de aprendizaje a 0,5 y a 1,1. ¿Converge? ¿Por qué?

**Ejercicio.** Repite el contraste de `d3_contraste.py` para la longitud del pico entre las especies Adelia y barbijo.